
# 🎨 ClawSouls — Avatar API Server (FastAPI + Cloudflared)

Sobe um servidor **FastAPI** no Colab que gera avatares por requisição HTTP.

**Fluxo:**
1. Rode este notebook (com GPU)
2. Copie a URL pública do Cloudflared
3. Envie POST `/generate` com os atributos da soul
4. Receba a imagem gerada em base64

---



## Pré-requisitos

- `Runtime > Change runtime type > T4 GPU`
- Não precisa do repositório clonado (tudo é self-contained)


In [10]:
# Toggle de modelo: escolha Z-Image-Turbo OU SD 1.5
USE_Z_IMAGE = False  # Mude para True para usar Z-Image-Turbo, False para SD 1.5

if USE_Z_IMAGE:
    MODEL_ID = "T5B/Z-Image-Turbo-FP8"
    TOTAL_STEPS = 8
    TOTAL_GUIDANCE = 0.0
else:
    MODEL_ID = "runwayml/stable-diffusion-v1-5"
    TOTAL_STEPS = 50
    TOTAL_GUIDance = 7.5


🔧 Modelo: turbo (T5B/Z-Image-Turbo-FP8)
   Steps: 8, Guidance: 0.0


In [11]:
# Toggle de modelo: escolha Z-Image-Turbo OU SD 1.5
USE_Z_IMAGE = False  # Mude para True para usar Z-Image-Turbo, False para SD 1.5

if USE_Z_IMAGE:
    MODEL_ID = "T5B/Z-Image-Turbo-FP8"
    TOTAL_STEPS = 8
    TOTAL_GUIDANCE = 0.0
else:
    MODEL_ID = "runwayml/stable-diffusion-v1-5"
    TOTAL_STEPS = 50
    TOTAL_GUIDance = 7.5


In [12]:

# Silenciar warnings
import warnings, os
warnings.filterwarnings("ignore", category=DeprecationWarning)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

!pip install -q fastapi uvicorn[standard] pydantic pillow
!pip install -q git+https://github.com/huggingface/diffusers transformers accelerate torch torchvision safetensors huggingface_hub

# cloudflared não está no PyPI — baixa o binário direto do GitHub
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print("✅ Dependências instaladas!")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Dependências instaladas!


In [13]:
# Toggle de modelo: escolha Z-Image-Turbo OU SD 1.5
USE_Z_IMAGE = False  # Mude para True para usar Z-Image-Turbo, False para SD 1.5

if USE_Z_IMAGE:
    MODEL_ID = "T5B/Z-Image-Turbo-FP8"
    TOTAL_STEPS = 8
    TOTAL_GUIDANCE = 0.0
else:
    MODEL_ID = "runwayml/stable-diffusion-v1-5"
    TOTAL_STEPS = 50
    TOTAL_GUIDance = 7.5


✅ server.py escrito em /content/


In [14]:
import subprocess
import time
import os

# Adiciona as variáveis de ambiente necessárias, incluindo HF_TOKEN
_env_vars = {
    'CLAWSOULS_MODELO': MODELO,
    'CLAWSOULS_SECRET': SECRET_TOKEN,
    'CLAWSOULS_STEPS': str(TOTAL_STEPS),
    'CLAWSOULS_GUIDANCE': str(TOTAL_GUIDANCE),
    'CLAWSOULS_MODEL_ID': MODEL_ID,
    'CLAWSOULS_WIDTH': str(cfg['width']),
    'CLAWSOULS_HEIGHT': str(cfg['height']),
    'CLAWSOULS_CROP_WIDTH': str(cfg.get('crop_width', 512)),
    'CLAWSOULS_CROP_HEIGHT': str(cfg.get('crop_height', 768)),
}

# Incluir HF_TOKEN no ambiente do subprocess, mesmo que vazio
_env_vars['HF_TOKEN'] = HF_TOKEN

log_file_path = '/tmp/server_logs.txt'

# Abre o arquivo de log para escrita e inicia o subprocesso
with open(log_file_path, 'w') as log_file:
    proc = subprocess.Popen(
        ['python', '/content/server.py'],
        env={**dict(__import__('os').environ), **_env_vars}, # Passa variáveis de ambiente
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True # Garante que a saída seja tratada como texto
    )

print('🚀 Servidor iniciado...')
time.sleep(7) # Aumenta o tempo de espera para dar mais chance de logs serem escritos e o servidor inicializar

# Imprimir logs do servidor
print('\n--- INÍCIO DOS LOGS DO SERVIDOR ---\n')
try:
    with open(log_file_path, 'r') as log_file:
        server_output = log_file.read()
        print(server_output)
except FileNotFoundError:
    print(f"⚠️  Arquivo de log '{log_file_path}' não encontrado.")
print('\n--- FIM DOS LOGS DO SERVIDOR ---\n')

# Testa healthcheck
import urllib.request
import json as _json
try:
    with urllib.request.urlopen('http://localhost:8000/health') as resp:
        health = _json.loads(resp.read().decode())
        print(f"✅ Health: {health}")
except Exception as e:
    print(f'⚠️  Healthcheck falhou: {e}')

🚀 Servidor iniciado...
⚠️  Healthcheck falhou: <urlopen error [Errno 111] Connection refused>


---

## Túnel Cloudflared


In [ ]:

import subprocess
import re

# Usa o token da conta Cloudflare fornecido
CLOUDFLARE_TOKEN = "eyJhIjoiNmIxNmYwMzUwNjM5NWFhNjBjZjk2NzY0MDA2Y2I0MGUiLCJ0IjoiYzA1MzQ2NGEtNzBhYy00MmUwLWFkMjQtZDdiMDFkYzBmOGZhIiwicyI6Ik1qbGtOekEyWVRjdFkyWXdOQzAwTXpGa0xXSTNZV1V0WkdJek5HVTJNRGhrT0RVeCJ9"

# Inicia cloudflared tunnel usando o token da conta
cloudflared_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', 'run', '--token', CLOUDFLARE_TOKEN],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Espera a URL do tunnel aparecer nos logs
tunnel_url = None
print('⏳ Esperando URL do tunnel...')
for i in range(30):
    line = cloudflared_proc.stdout.readline().decode('utf-8', errors='replace')
    if not line:
        time.sleep(0.5)
        continue
    print(line.strip())
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print()
    print('=' * 60)
    print('🌐 TÚNEL ATIVO!')
    print(f'📎 URL pública: {tunnel_url}')
    print()
    print('Endpoints:')
    print(f'  GET  {tunnel_url}/health')
    print(f'  GET  {tunnel_url}/models')
    print(f'  POST {tunnel_url}/generate')
    print('=' * 60)
    print()
    print('⚠️  TOKEN: ' + SECRET_TOKEN)
    print()
    print('📋 Copie esta URL e cole aqui no chat para eu usar!')
else:
    print('⚠️  Não foi possível obter a URL do tunnel.')
    print('   Verifique: cloudflared_proc.wait()')


⏳ Esperando URL do tunnel...
2026-05-09T23:54:47Z INF Starting tunnel tunnelID=c053464a-70ac-42e0-ad24-d7b01dc0f8fa
2026-05-09T23:54:47Z INF Version 2026.3.0 (Checksum 4a9e50e6d6d798e90fcd01933151a90bf7edd99a0a55c28ad18f2e16263a5c30)
2026-05-09T23:54:47Z INF GOOS: linux, GOVersion: go1.24.13, GoArch: amd64
2026-05-09T23:54:47Z INF Settings: map[token:*****]
2026-05-09T23:54:47Z INF Autoupdate frequency is set autoupdateFreq=86400000
2026-05-09T23:54:47Z INF Generated Connector ID: 646e7a0f-4748-4a7f-96c3-bf5b3cf266cb
2026-05-09T23:54:47Z INF Initial protocol quic
2026-05-09T23:54:47Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
2026-05-09T23:54:47Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
2026-05-09T23:54:47Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
2026-05-09T23:54:47Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
2026-05-09T23:54:47Z INF Starting metrics server on 127.0.0.1:20241/metrics
2026-05-09T23:54:47Z INF Tunnel connection c

---

## Como usar

Cole a URL do tunnel aqui no chat. Eu monto o prompt e faço a requisição!

**Exemplo mínimo:**

```
POST /generate?token=cs-secret-2026
{"custom_prompt": "a cyberpunk mage with bionic arms, neon glow, detailed face"}
```

**Exemplo completo:**

```
POST /generate?token=cs-secret-2026
{"name": "Mage Cyberpunk", "custom_prompt": "a grizzled 60-year-old mage..."}
```

**Notas:**
- Turbo (default): 1024×1024 gerado, cropado para 512×768 bust portrait
- ~2-4 segundos por imagem
- Sem guidance (CFG=0) — prompt deve ser bem descritivo
